# Risque de rupture de stock : couverture en jours

**Objectif :** repérer les produits achetés dont le stock risque de ne pas tenir jusqu'à la prochaine livraison fournisseur. On compare la **couverture de stock** (stock actuel ÷ ventes moyennes par jour) au **délai moyen du fournisseur** (`avg_lead_time`).

**Périmètre :** produits finis achetés à l'extérieur (`finishedgoodsflag = 1` et `makeflag = 0`), les seuls concernés par le risque fournisseur. Pour les produits fabriqués en interne, l'approvisionnement ne dépend pas d'un fournisseur.

**Démarche :** on construit d'abord le stock actuel, on constate que la classification n'est pas exploitable (5 produits seulement sous leur point de commande), puis on calcule la couverture en jours.

In [1]:
import sqlite3
import numpy as np
import pandas as pd

# Affiche toutes les colonnes des DataFrame (pas de "..." au milieu)
pd.set_option("display.max_columns", None)

# Connexion à la base SQLite du projet (chemin relatif depuis le dossier ml/)
conn = sqlite3.connect("../database/adventureworks.db")

## 1. Base : stock actuel et point de commande

Le stock d'un produit est la **somme de ses quantités dans tous les entrepôts** (un produit peut être réparti sur plusieurs emplacements). On garde les produits finis (`finishedgoodsflag = 1`) achetés (`makeflag = 0`).

In [2]:
# Requête de base : un produit fini acheté par ligne, avec son stock total (tous entrepôts confondus)
# LEFT JOIN : on garde aussi un éventuel produit sans ligne d'inventaire (stock = NaN)
query_base = """
SELECT
    p.productid,
    p.name,
    p.makeflag,
    p.reorderpoint,
    p.safetystocklevel,
    s.current_quantity
FROM product p
LEFT JOIN (
    SELECT productid, SUM(quantity) AS current_quantity   -- stock total, tous entrepôts confondus
    FROM productinventory
    GROUP BY productid
) s ON s.productid = p.productid
WHERE p.finishedgoodsflag = 1
  AND p.makeflag = 0          -- produits achetés : seuls concernés par le risque fournisseur
"""

df = pd.read_sql(query_base, conn)
print(f"Nombre de produits (échantillons) : {df.shape[0]}")
df.head()

Nombre de produits (échantillons) : 83


,productid,name,makeflag,reorderpoint,safetystocklevel,current_quantity
0,707,"Sport-100 Helmet, Red",0,3,4,288
1,708,"Sport-100 Helmet, Black",0,3,4,324
2,709,"Mountain Bike Socks, M",0,3,4,180
3,710,"Mountain Bike Socks, L",0,3,4,216
4,711,"Sport-100 Helmet, Blue",0,3,4,216


In [3]:
df["label_rupture"] = (df["current_quantity"] < df["reorderpoint"]).astype(int)
print(df["label_rupture"].value_counts())
print("Produits sans stock enregistré :", df["current_quantity"].isna().sum())

label_rupture
0    78
1     5
Name: count, dtype: int64
Produits sans stock enregistré : 0


**Constat :** seuls 5 produits sur 83 (6 %) sont sous leur point de commande. Avec si peu de cas positifs, une classification (rupture oui/non) n'est pas exploitable. On passe à un indicateur de **couverture en jours**, qui tient compte de la vitesse de vente.

## 2. Vitesse de vente

Ventes moyennes par jour = quantité totale vendue ÷ nombre de jours entre la première et la dernière vente (au minimum 1 jour).

In [4]:
query_ventes = """
SELECT
    sod.productid,
    SUM(sod.orderqty) AS total_vendu,
    CAST(julianday(MAX(soh.orderdate)) - julianday(MIN(soh.orderdate)) AS INTEGER) + 1 AS nb_jours
FROM salesorderdetail sod
JOIN salesorderheader soh ON soh.salesorderid = sod.salesorderid
GROUP BY sod.productid
"""
ventes = pd.read_sql(query_ventes, conn)
ventes["avg_daily_sales"] = ventes["total_vendu"] / ventes["nb_jours"]
ventes.describe().round(2)

,productid,total_vendu,nb_jours,avg_daily_sales
count,266.00,266.00,266.00,266.00
mean,857.64,1033.51,468.42,2.09
std,84.86,1221.85,247.51,2.03
min,707.00,4.00,1.00,0.07
25%,783.25,372.75,337.00,0.98
50%,863.50,643.00,365.00,1.56
75%,931.75,1136.75,640.00,2.37
max,999.00,8311.00,1127.00,17.17


## 3. Délai moyen du fournisseur

Le délai moyen (`averageleadtime`, en jours) vient de la table `ProductVendor` ; si un produit a plusieurs fournisseurs, on prend la moyenne.

In [5]:
query_delai = """
SELECT productid, AVG(averageleadtime) AS avg_lead_time
FROM productvendor
GROUP BY productid
"""
delai = pd.read_sql(query_delai, conn)
delai.describe().round(2)

,productid,avg_lead_time
count,265.00,265.00
mean,559.51,21.27
std,221.21,11.84
min,1.00,10.00
25%,397.00,16.00
50%,467.00,17.33
75%,847.00,18.00
max,952.00,120.00


## 4. Couverture de stock en jours

Couverture = stock actuel ÷ ventes moyennes par jour. Un produit est **à risque** si sa couverture est inférieure au délai moyen du fournisseur : le stock ne tiendra pas jusqu'à la prochaine livraison.

In [6]:
df = df.drop(columns=["avg_daily_sales", "avg_lead_time", "jours_couverture", "a_risque"], errors="ignore")

df = df.merge(ventes[["productid", "avg_daily_sales"]], on="productid", how="left")
df = df.merge(delai, on="productid", how="left")

print("Produits sans vente :", df["avg_daily_sales"].isna().sum())
print("Produits sans délai fournisseur :", df["avg_lead_time"].isna().sum())

df["jours_couverture"] = df["current_quantity"] / df["avg_daily_sales"]
df["a_risque"] = df["jours_couverture"] < df["avg_lead_time"]

print("Produits à risque :", df["a_risque"].sum(), "sur", df["jours_couverture"].notna().sum(), "produits évaluables")

df[["name", "current_quantity", "avg_daily_sales", "jours_couverture", "avg_lead_time", "a_risque"]] \
    .sort_values("jours_couverture").head(10).round(1)

Produits sans vente : 7
Produits sans délai fournisseur : 0
Produits à risque : 19 sur 76 produits évaluables


,name,current_quantity,avg_daily_sales,jours_couverture,avg_lead_time,a_risque
28,"Half-Finger Gloves, M",0,4.5,0.0,15.0,True
22,"Women's Tights, M",0,1.2,0.0,60.0,True
51,"Short-Sleeve Classic Jersey, M",0,1.0,0.0,30.0,True
45,Hitch Rack - 4-Bike,0,8.0,0.0,45.0,True
46,Bike Wash - Dissolver,36,8.4,4.3,45.0,True
52,"Short-Sleeve Classic Jersey, L",36,7.2,5.0,30.0,True
53,"Short-Sleeve Classic Jersey, XL",72,9.7,7.4,30.0,True
23,"Women's Tights, L",36,3.2,11.4,60.0,True
39,Water Bottle - 30 oz.,252,17.2,14.7,45.0,True
49,Hydration Pack - 70 oz.,108,7.0,15.5,45.0,True


## 5. Produits à risque : quantité manquante

Pour chaque produit à risque, on estime la quantité qui manquera avant la prochaine livraison : ventes moyennes × délai − stock actuel.

In [7]:
risque = df[df["a_risque"]].copy()
# quantité manquante pour tenir jusqu'à la livraison = ventes pendant le délai - stock actuel
risque["manque_pendant_delai"] = np.ceil(risque["avg_daily_sales"] * risque["avg_lead_time"] - risque["current_quantity"])

risque[["name", "current_quantity", "avg_daily_sales", "jours_couverture", "avg_lead_time", "manque_pendant_delai"]] \
    .sort_values("jours_couverture").round(1)

,name,current_quantity,avg_daily_sales,jours_couverture,avg_lead_time,manque_pendant_delai
22,"Women's Tights, M",0,1.2,0.0,60.0,71.0
28,"Half-Finger Gloves, M",0,4.5,0.0,15.0,69.0
45,Hitch Rack - 4-Bike,0,8.0,0.0,45.0,359.0
51,"Short-Sleeve Classic Jersey, M",0,1.0,0.0,30.0,31.0
46,Bike Wash - Dissolver,36,8.4,4.3,45.0,341.0
52,"Short-Sleeve Classic Jersey, L",36,7.2,5.0,30.0,181.0
53,"Short-Sleeve Classic Jersey, XL",72,9.7,7.4,30.0,220.0
23,"Women's Tights, L",36,3.2,11.4,60.0,155.0
39,Water Bottle - 30 oz.,252,17.2,14.7,45.0,521.0
49,Hydration Pack - 70 oz.,108,7.0,15.5,45.0,205.0


## 6. Lien avec les segments fournisseurs

On compte les lignes d'achat de chaque fournisseur concerné. En dessous de 30 lignes, un fournisseur n'est pas évalué par le clustering (segment « Peu d'historique »).

In [8]:

q_vendor = """
SELECT pv.productid, pv.businessentityid AS vendorid,
       COUNT(pod.purchaseorderdetailid) AS nb_lignes
FROM productvendor pv
LEFT JOIN purchaseorderheader poh ON poh.vendorid = pv.businessentityid
LEFT JOIN purchaseorderdetail pod ON pod.purchaseorderid = poh.purchaseorderid
GROUP BY pv.productid, pv.businessentityid
"""
pv_lignes = pd.read_sql(q_vendor, conn)

lien = risque[["productid", "name", "jours_couverture", "avg_lead_time"]].merge(
    pv_lignes, on="productid", how="left")
lien["segment"] = lien["nb_lignes"].apply(
    lambda n: "Peu d'historique" if n < 30 else "Évalué par le clustering")

print(lien["segment"].value_counts())
print("\nFournisseurs concernés :", lien["vendorid"].nunique())
display(lien.groupby("vendorid").agg(nb_produits=("productid", "count"),
                                      nb_lignes=("nb_lignes", "first")))

segment
Peu d'historique    19
Name: count, dtype: int64

Fournisseurs concernés : 5


,nb_produits,nb_lignes
vendorid,,
1520,1,3
1546,6,9
1594,5,9
1636,5,10
1676,2,15


## Conclusion

- Sur 83 produits externes, seuls **5** sont sous leur `reorderpoint` : le seuil fixe ne permet pas de construire un modèle de classification.
- L'indicateur de **couverture en jours** (stock / ventes journalières vs délai fournisseur) signale **19 produits à risque sur 76**, soit près de 4 fois plus que le seuil fixe.
- Ces 19 produits proviennent de **5 fournisseurs** seulement, tous classés « Peu d'historique » (moins de 30 lignes d'achat) : ils ne sont donc **pas évalués par le clustering**.
- Parmi eux, le vendorid 1636, déjà associé aux ventes à perte (AWC Logo Cap, Jerseys).
- **Limites** : ventes moyennes sans saisonnalité, stock tous entrepôts confondus, commandes en cours ignorées, 7 produits sans vente exclus (couverture non calculable), 72 produits finis sans enregistrement de stock hors périmètre.

In [9]:
conn.close()